# Goal + Intake Rule (MVP)

## Decision outputs
- `answer` | `clarify` | `escalate`

## Intake rule (when we have enough context to proceed)
Proceed only if we have:
- At least one identifier: `order_id` **or** `purchase_date` **or** `email`
- Plus an issue summary: `issue_summary` (what actually went wrong)

If anything is missing, respond with `clarify` and ask **one** concise clarifying question (max 2 asks).

## Safety / privacy
- Never ask for: passwords, OTPs, full card numbers, CVV
- Never guess order/account details; if unsure, clarify or escalate


In [32]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Optional
from pydantic import BaseModel
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [33]:
load_dotenv()

True

In [34]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [35]:
from __future__ import annotations

from typing import Literal, Optional, TypedDict

Decision = Literal["answer", "clarify", "escalate"]


class Message(TypedDict):
    role: Literal["user", "assistant"]
    content: str


class Intake(TypedDict, total=False):
    order_id: Optional[str]
    purchase_date: Optional[str]
    email: Optional[str]
    issue_summary: Optional[str]


class ConversationState(TypedDict, total=False):
    messages: list[Message]
    intake: Intake
    decision: Decision
    missing_fields: list[str]


In [36]:
def get_missing_fields(intake: Intake) -> list[str]:
    missing: list[str] = []
    if not intake.get("order_id") and not intake.get("email") and not intake.get("purchase_date"):
        missing.append("identifier")
    if not intake.get("issue_summary"):
        missing.append("issue_summary")
    return missing


def make_clarifying_question(missing_fields: list[str]) -> str:
    if "identifier" in missing_fields and "issue_summary" in missing_fields:
        return (
            "Could you please provide your order ID, email, or purchase date, "
            "as well as a brief summary of the issue you're facing?"
        )
    if "identifier" in missing_fields:
        return "Could you please provide your order ID, email, or purchase date to help us identify your purchase?"
    if "issue_summary" in missing_fields:
        return "Could you please provide a brief summary of the issue you're facing with your order?"
    return ""


def validate_intake(intake: Intake) -> tuple[Literal["answer", "clarify"], list[str], str | None]:
    missing_fields = get_missing_fields(intake)
    if missing_fields:
        return "clarify", missing_fields, make_clarifying_question(missing_fields)
    return "answer", missing_fields, None


In [ ]:
import re

EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
ORDER_RE = re.compile(r"(?:order\s*#?\s*|#)\s*([A-Za-z0-9-]{4,})", re.IGNORECASE)
DATE_RE = re.compile(r"\b(\d{4}-\d{2}-\d{2})\b")  # YYYY-MM-DD

ISSUE_HINTS = (
    "refund",
    "return",
    "cancel",
    "broken",
    "damaged",
    "wrong",
    "missing",
    "can't",
    "cant",
    "won't",
    "wont",
    "not working",
    "error",
    "problem",
)


def extract_fields(user_text: str) -> Intake:
    text = user_text.strip()
    intake: Intake = {}

    m = EMAIL_RE.search(text)
    if m:
        intake["email"] = m.group(0)

    m = ORDER_RE.search(text)
    if m:
        intake["order_id"] = m.group(1)

    m = DATE_RE.search(text)
    if m:
        intake["purchase_date"] = m.group(1)

    lower = text.lower()
    if any(h in lower for h in ISSUE_HINTS):
        intake["issue_summary"] = text

    return intake


def merge_intake(existing: Intake, new: Intake) -> Intake:
    merged: Intake = dict(existing)
    for k, v in new.items():
        if v and not merged.get(k):
            merged[k] = v
    return merged


state: ConversationState = {
    "messages": [],
    "intake": {},
    "decision": "clarify",
    "missing_fields": ["identifier", "issue_summary"],
}

print("Type /exit to quit.")
while True:
    user_text = input("> ").strip()
    if user_text in ("/exit", "/quit"):
        break

    state["messages"].append({"role": "user", "content": user_text})

    extracted = extract_fields(user_text)
    state["intake"] = merge_intake(state.get("intake", {}), extracted)

    decision, missing, question = validate_intake(state["intake"])
    state["decision"] = decision
    state["missing_fields"] = missing

    if decision == "clarify":
        assistant_text = question or "Can you share a bit more detail?"
        print(assistant_text)
        state["messages"].append({"role": "assistant", "content": assistant_text})
        continue

    assistant_text = "Thanks—this is enough context to proceed."
    print(assistant_text)
    state["messages"].append({"role": "assistant", "content": assistant_text})


IndentationError: unexpected indent (3915400576.py, line 12)